# AAVS1 (PPP1R12C) — 378 bp CMV cassette integration (K562)

Reproduces the MCP `simulate_integration` call. Inserts a
construct at **chr19:55115000** (no genomic bases removed)
and scores regulatory disruption at the surrounding locus.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.integration import simulate_integration
from chorus.analysis.analysis_request import AnalysisRequest

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs.
oracle_name = 'alphagenome'
position = 'chr19:55115000'
gene_name = 'PPP1R12C'
assay_ids = [
    'DNASE/EFO:0002067 DNase-seq/.',
    'CHIP_HISTONE/EFO:0002067 Histone ChIP-seq H3K27ac/.',
    'CAGE/hCAGE EFO:0002067/+',
]
construct_sequence = (
        "TAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCCCATATATGGAGTTCCGCGTT"
        "ACATAACTTACGGTAAATGGCCCGCCTGGCTGACCGCCCAACGACCCCCGCCCATTGACGTCAA"
        "TAATGACGTATGTTCCCATAGTAACGCCAATAGGGACTTTCCATTGACGTCAATGGGTGGAGTA"
        "TTTACGGTAAACTGCCCACTTGGCAGTACATCAAGTGTATCATATGCCAAGTACGCCCCCTATT"
        "GACGTCAATGACGGTAAATGGCCCGCCTGGCATTATGCCCAGTACATGACCTTATGGGACTTTC"
        "CTACTTGGCAGTACATCTACGTATTAGTCATCGCTATTACCATGGTGATGCGGTTTTG"
    )
print(f"Construct length: {len(construct_sequence)} bp")

In [ ]:
# Score the integration. Same scorer as analyze_region_swap but
# semantically an insertion (genomic bases retained).
normalizer = get_normalizer(oracle_name=oracle_name)
analysis_request = AnalysisRequest(
    user_prompt=(
        f"Insert a {len(construct_sequence)} bp construct at {position} "
        "and predict local chromatin disruption."
    ),
    tool_name="simulate_integration",
    oracle_name=oracle_name,
    tracks_requested=f"{len(assay_ids)} K562 tracks",
)
report = simulate_integration(
    oracle=oracle,
    position=position,
    construct_sequence=construct_sequence,
    assay_ids=assay_ids,
    gene_name=gene_name,
    normalizer=normalizer,
    oracle_name=oracle_name,
)
report.analysis_request = analysis_request

In [ ]:
# Save the same artifacts the MCP tool would produce:
#   - example_output.md  (markdown report)
#   - example_output.json (structured scores)
#   - example_output.tsv (track-level table)
#   - integration_CMV_PPP1R12C_report.html (interactive IGV report)
WALKTHROUGH_DIR.joinpath("example_output.md").write_text(report.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(report.to_dict(), indent=2, default=str)
)
try:
    report.to_dataframe().to_csv(
        WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
    )
except Exception as exc:
    print(f"TSV write skipped: {exc}")

_html_path = report.to_html(output_path=str(WALKTHROUGH_DIR / "integration_CMV_PPP1R12C_report.html"))
print(f"Wrote artifacts to { WALKTHROUGH_DIR }")


## What this notebook produced

- `example_output.md/.json/.tsv` — WT vs integration per-layer scores
- `integration_CMV_PPP1R12C_report.html` — IGV browser overlay
